# Simulate a Solid Bunny Deforming Under Gravity

In [ ]:
import sys; sys.path.append('..')
import MeshFEM
import mesh, elastic_solid, energy, sim_utils, loads, benchmark, py_newton_optimizer
import tri_mesh_viewer

In [ ]:
m = mesh.Mesh('../../misc/examples/meshes/bunny_coarse.msh', degree=1)
es = elastic_solid.ElasticSolid(m, energy.NeoHookeanYoungPoisson(3, E=2, nu=0.4)) # Silicon material (Y= 2MPa, nu=0.4)
g = loads.Gravity(es, rho=1e-4, g=[0, -9.81, 0]) # mass density in kg/mm^3, gravitational acceleration in N/kg

In [ ]:
RECORD_VIDEO=True

In [ ]:
v = tri_mesh_viewer.Viewer(es, wireframe=True, offscreen=RECORD_VIDEO)
v.makeOpaque(color='#48B3FF')
v.tetShrinkFactor = 0.5
v.show()

In [ ]:
# Glue to the ground.
fixedVars = sim_utils.getBBoxVars(es, sim_utils.BBoxFace.MIN_Y, tol=4)

In [ ]:
# Configure some options of the Newton solver.
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.factorizer = opts.factorizer.CatamariNesdis # This is also the default.
opts.hessianUpdateController = py_newton_optimizer.HessianUpdatePeriodic()
opts.hessianUpdateController.period = 0 # `period` is the number of iterations *between* factorizations.
opts.hessianProjectionController = py_newton_optimizer.HessianProjectionNever()

In [ ]:
benchmark.reset()
if RECORD_VIDEO: v.recordStart('bunny.mp4', outWidth=512, outHeight=512)
es.computeEquilibrium(loads=[g], fixedVars=fixedVars, opts=opts, cb=tri_mesh_viewer.ViewUpdater(v, showStress=False))
if RECORD_VIDEO: v.recordStop()
benchmark.report()